### Regex notes:

\w : Word character (A-Z, a-z, 0-9, and underscore).

\s : Whitespace (space, tab, newline).

. : Any single character (except a newline).

\d : Digit. Matches any single number from 0 to 9.

Capital letters negate them: \D (Not a digit), \W (Not a word char), \S (Not whitespace).


### Quantifiers (how many?):

Quantifiers (How many?):

* : Zero or more times.

+ : One or more times.

? : Zero or one time (optional).

{3} : Exactly 3 times.


### Anchors & Groups (Where and what?):

^ : Start of the string.

$ : End of the string.

[A-Z] : Character set (matches any one uppercase letter).

(...) : Capture group (extracts exactly what is inside the parentheses).


Pandas has built-in regex functions under the .str accessor. The most common use case is extracting specific data from a messy text column into a clean new column using str.extract().

Note: str.extract() requires you to use parentheses () in your regex to tell it exactly which part to extract.



In [5]:
import pandas as pd
import numpy as np

# Sample dataframe
df = pd.DataFrame({'col1': ['Invoice 104', 'Order 59', 'Ticket 8023', 'Ticket Master', 'I love Tikcet', 'You mispelled Ticket', 'lower case ticket?']})

# Extract just the numbers from 'col1' and put them in 'col2'
df['col2'] = df['col1'].str.extract(r'(\d+)')

print(df)

                   col1  col2
0           Invoice 104   104
1              Order 59    59
2           Ticket 8023  8023
3         Ticket Master   NaN
4         I love Tikcet   NaN
5  You mispelled Ticket   NaN
6    lower case ticket?   NaN


### Check if pattern exists just once

In [6]:
df['col3'] = np.where(df['col1'].str.contains(r'Ticket'), 'abc', 'ok')
df.head()

,col1,col2,col3
0,Invoice 104,104,ok
1,Order 59,59,ok
2,Ticket 8023,8023,abc
3,Ticket Master,NaN,abc
4,I love Tikcet,NaN,ok


### Counting the number of instances:

In [7]:
df = pd.DataFrame({'col1': ['A B A C A', 'B C', 'A A']})

# Count how many times the letter 'A' appears in 'col1'
df['col2'] = df['col1'].str.count(r'A')

print(df)

        col1  col2
0  A B A C A     3
1        B C     0
2        A A     2


### Multiple conditions existence:

In [8]:
# Sample dataframe with various scenarios
df = pd.DataFrame({'col1': [
    'System Error but also a Success',     # Has both
    'Error: Timeout',                      # Just Error
    'Success Success Success Success',     # 4 Successes
    'Done with Success',                   # 1 Success
    'Pending status'                       # Neither
]})

# 1. Define conditions (Most specific at the top!)
conditions = [
    # 1st: Has BOTH 'Error' AND 'Success'
    df['col1'].str.contains(r'Error') & df['col1'].str.contains(r'Success'),
    
    # 2nd: Has 'Error' (but not Success, because that would have been caught above)
    df['col1'].str.contains(r'Error'),
    
    # 3rd: Has strictly MORE than 3 instances of 'Success'
    df['col1'].str.count(r'Success') > 3,
    
    # 4th: Has 'Success' (1 to 3 times, because >3 was caught above)
    df['col1'].str.contains(r'Success')
]

# 2. Define choices in the exact same order
choices = [
    'Confusion!',
    'Error',
    'Super success',
    'Success'
]

# 3. Apply the logic
df['col2'] = np.select(conditions, choices, default='Other')

print(df)

                              col1           col2
0  System Error but also a Success     Confusion!
1                   Error: Timeout          Error
2  Success Success Success Success  Super success
3                Done with Success        Success
4                   Pending status          Other
